<a href="https://colab.research.google.com/github/MDD111222333/KT3_qa-testing-coursework/blob/main/%D0%A2%D0%9F%D0%A2_%D0%9A%D0%A23_3%D0%9F3_%D0%9C%D0%BE%D1%80%D0%BE%D0%B7%D0%BE%D0%B2%D0%94%D0%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install pytest

In [9]:
%%file logic.py
import math

class TaskLogic:
    @staticmethod
    def is_even(number: int) -> bool:
        # bool в Python наследуется от int (True == 1), поэтому исключаем его явно
        if type(number) is not int:
            raise TypeError("Аргумент должен быть целым числом (int)")
        return number % 2 == 0

    @staticmethod
    def calculate_area(length: float, width: float) -> float:
        if type(length) in (bool, str) or type(width) in (bool, str) or not isinstance(length, (int, float)) or not isinstance(width, (int, float)):
            raise TypeError("Длина и ширина должны быть числами")
        if math.isnan(length) or math.isnan(width) or math.isinf(length) or math.isinf(width):
            raise ValueError("Длина и ширина не могут быть NaN или Infinity")
        if length < 0 or width < 0:
            raise ValueError("Длина и ширина не могут быть отрицательными")
        return length * width

    @staticmethod
    def classify_triangle(a: float, b: float, c: float) -> str:
        for side in (a, b, c):
            if type(side) in (bool, str) or not isinstance(side, (int, float)):
                raise TypeError("Стороны должны быть числами")
            if math.isnan(side) or math.isinf(side):
                raise ValueError("Стороны не могут быть NaN или Infinity")
            if side <= 0:
                raise ValueError("Длины сторон должны быть больше нуля")

        if (a + b <= c) or (a + c <= b) or (b + c <= a):
            raise ValueError("Стороны не образуют треугольник")

        if a == b == c:
            return "равносторонний"
        elif a == b or b == c or a == c:
            return "равнобедренный"
        else:
            return "разносторонний"

    @staticmethod
    def calculate_discount(price: float, discount: float) -> float:
        if type(price) in (bool, str) or type(discount) in (bool, str) or not isinstance(price, (int, float)) or not isinstance(discount, (int, float)):
            raise TypeError("Цена и скидка должны быть числами")
        if math.isnan(price) or math.isnan(discount) or math.isinf(price) or math.isinf(discount):
            raise ValueError("Значения не могут быть NaN или Infinity")
        if price < 0 or discount < 0 or discount > 100:
            raise ValueError("Некорректная цена или процент скидки")
        return price * (1 - discount / 100)

Overwriting logic.py


In [11]:
%%file test_task1.py
import pytest
from logic import TaskLogic

# Позитивные тесты (включая предельно большие целые числа)
@pytest.mark.parametrize("number, expected", [
    (2, True),
    (0, True),
    (-4, True),
    (1, False),
    (10**100, True),
    (10**100 + 1, False),
])
def test_is_even_valid(number, expected):
    assert TaskLogic.is_even(number) == expected

# Негативные тесты: Символы, строки, float, NaN, Inf, bool
@pytest.mark.parametrize("invalid_input", [
    "2",
    "abc",
    "!@#$%^&*",
    "",
    2.0,
    2.5,
    float("inf"),
    float("nan"),
    True,
    None,
    [2],
])
def test_is_even_invalid_type(invalid_input):
    with pytest.raises(TypeError):
        TaskLogic.is_even(invalid_input)

Writing test_task1.py


In [12]:
!pytest test_task1.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 17 items                                                             

test_task1.py::test_is_even_valid[2-True] PASSED                         [  5%]
test_task1.py::test_is_even_valid[0-True] PASSED                         [ 11%]
test_task1.py::test_is_even_valid[-4-True] PASSED                        [ 17%]
test_task1.py::test_is_even_valid[1-False] PASSED                        [ 23%]
test_task1.py::test_is_even_valid[10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000-True] PASSED [ 29%]
test_task1.py::test_is_even_valid[10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001-False] PASSED [ 35%]
test_task1.py::test_is_ev

In [13]:
%%file test_task2.py
import pytest
from logic import TaskLogic

@pytest.mark.parametrize("length, width, expected", [
    (5, 10, 50),
    (3.5, 2.0, 7.0),
    (0.0001, 0.0002, 0.00000002),
    (10**50, 10**50, 10**100),
])
def test_calculate_area_valid(length, width, expected):
    assert pytest.approx(TaskLogic.calculate_area(length, width)) == expected

@pytest.mark.parametrize("length, width", [
    ("10", 5), (5, "abc"), ("!@#", "$%^"), (True, 10),
])
def test_calculate_area_invalid_types(length, width):
    with pytest.raises(TypeError):
        TaskLogic.calculate_area(length, width)

@pytest.mark.parametrize("length, width", [
    (-5, 10), (float("inf"), 10), (10, float("nan")),
])
def test_calculate_area_invalid_values(length, width):
    with pytest.raises(ValueError):
        TaskLogic.calculate_area(length, width)

Writing test_task2.py


In [14]:
!pytest test_task2.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 11 items                                                             

test_task2.py::test_calculate_area_valid[5-10-50] PASSED                 [  9%]
test_task2.py::test_calculate_area_valid[3.5-2.0-7.0] PASSED             [ 18%]
test_task2.py::test_calculate_area_valid[0.0001-0.0002-2e-08] PASSED     [ 27%]
test_task2.py::test_calculate_area_valid[100000000000000000000000000000000000000000000000000-100000000000000000000000000000000000000000000000000-10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000] PASSED [ 36%]
test_task2.py::test_calculate_area_invalid_types[10-5] PASSED            [ 45%]
test_task2.py::test_calculate_area_invalid_types[5-abc] PASSED           [ 5

In [15]:
%%file test_task3.py
import pytest
from logic import TaskLogic

@pytest.mark.parametrize("a, b, c, expected", [
    (3, 3, 3, "равносторонний"),
    (2.5, 2.5, 4.0, "равнобедренный"),
    (3.1, 4.2, 5.3, "разносторонний"),
    (10**50, 10**50, 10**50, "равносторонний"),
])
def test_classify_triangle_valid(a, b, c, expected):
    assert TaskLogic.classify_triangle(a, b, c) == expected

@pytest.mark.parametrize("a, b, c", [
    ("3", 3, 3), ("abc", "def", "ghi"), ("!@#", 4, 5), (None, 2, 3),
])
def test_classify_triangle_invalid_types(a, b, c):
    with pytest.raises(TypeError):
        TaskLogic.classify_triangle(a, b, c)

@pytest.mark.parametrize("a, b, c", [
    (0, 3, 3), (-1, 2, 2), (1, 2, 3), (float("inf"), 5, 5), (float("nan"), 5, 5),
])
def test_classify_triangle_invalid_values(a, b, c):
    with pytest.raises(ValueError):
        TaskLogic.classify_triangle(a, b, c)

Writing test_task3.py


In [16]:
!pytest test_task3.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 13 items                                                             

test_task3.py::test_classify_triangle_valid[3-3-3-\u0440\u0430\u0432\u043d\u043e\u0441\u0442\u043e\u0440\u043e\u043d\u043d\u0438\u0439] PASSED [  7%]
test_task3.py::test_classify_triangle_valid[2.5-2.5-4.0-\u0440\u0430\u0432\u043d\u043e\u0431\u0435\u0434\u0440\u0435\u043d\u043d\u044b\u0439] PASSED [ 15%]
test_task3.py::test_classify_triangle_valid[3.1-4.2-5.3-\u0440\u0430\u0437\u043d\u043e\u0441\u0442\u043e\u0440\u043e\u043d\u043d\u0438\u0439] PASSED [ 23%]
test_task3.py::test_classify_triangle_valid[100000000000000000000000000000000000000000000000000-100000000000000000000000000000000000000000000000000-100000000000000000000000000000000000000000000000

In [17]:
%%file test_task4.py
import pytest
from logic import TaskLogic

@pytest.mark.parametrize("price, discount, expected", [
    (1000, 10, 900.0),
    (199.99, 15.5, 168.99155),
    (10**12, 50, 5 * 10**11),
])
def test_calculate_discount_valid(price, discount, expected):
    assert pytest.approx(TaskLogic.calculate_discount(price, discount)) == expected

@pytest.mark.parametrize("price, discount", [
    ("1000", 10), (1000, "10%"), ("free", "!@#"),
])
def test_calculate_discount_invalid_types(price, discount):
    with pytest.raises(TypeError):
        TaskLogic.calculate_discount(price, discount)

@pytest.mark.parametrize("price, discount", [
    (-100, 10), (100, -5), (100, 105), (float("nan"), 10), (100, float("inf")),
])
def test_calculate_discount_invalid_values(price, discount):
    with pytest.raises(ValueError):
        TaskLogic.calculate_discount(price, discount)

Writing test_task4.py


In [18]:
!pytest test_task4.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 11 items                                                             

test_task4.py::test_calculate_discount_valid[1000-10-900.0] PASSED       [  9%]
test_task4.py::test_calculate_discount_valid[199.99-15.5-168.99155] PASSED [ 18%]
test_task4.py::test_calculate_discount_valid[1000000000000-50-500000000000] PASSED [ 27%]
test_task4.py::test_calculate_discount_invalid_types[1000-10] PASSED     [ 36%]
test_task4.py::test_calculate_discount_invalid_types[1000-10%] PASSED    [ 45%]
test_task4.py::test_calculate_discount_invalid_types[free-!@#] PASSED    [ 54%]
test_task4.py::test_calculate_discount_invalid_values[-100-10] PASSED    [ 63%]
test_task4.py::test_calculate_discount_invalid_values[100--5] PASSED     [ 72%]
test_t